# EHR Pipeline Benchmark on MIMIC-III-Ext-Notes

Benchmarks the `ehr_pipeline` package on real MIMIC-III clinical notes joined with the matching structured EHR.

Inputs:
- `datasets/mimic-iii-ext-notes-1.0.0/notes.csv` — 150 deidentified clinical notes (`row_id`, `subject_id`, `hadm_id`, `text`).
- `datasets/mimic-iii-ext-notes-1.0.0/labels.csv` — 2,288 expert-annotated clinical concepts per note, used as the **gold entity set** (filtered to `detection=yes`, `encounter=yes`, `negation=no`).
- `datasets/bquxjob_5dbe3bd2_19dae30f3cf.json` — structured EHR (diagnoses + procedures) per `hadm_id`. 130 of 130 unique note `hadm_id`s match.

Per case the notebook:
1. Materializes a FHIR R4 Bundle from the structured EHR and writes the **real** clinical note text to disk.
2. Runs the 8-stage pipeline (the context agent / stage 4 is **disabled** by default for benchmarking — `ENABLE_CONTEXT_AGENT`).
3. Builds a deterministic reference summary from gold concepts + structured EHR fields.
4. Scores the generated Markdown against the reference with **ROUGE-1/2/L**, **BERTScore P/R/F1**, **entity precision / recall / F1**, and reports the achieved **compression ratio** vs the 0.20–0.30 target.

Setup: `pip install -r requirements-bench.txt`. The pipeline reads `OLLAMA_API_KEY` and `OLLAMA_HOST` from `.env`.

In [22]:
import importlib
import logging
import re
import sys
import warnings
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
warnings.filterwarnings("ignore", category=FutureWarning)

# Force-reload every ehr_pipeline submodule so stale model names are never used.
# Re-run this cell if you edit config.py, prompts.py, or any stage file.
import ehr_pipeline.config
import ehr_pipeline.prompts
import ehr_pipeline.ollama_client
import ehr_pipeline.schemas
import ehr_pipeline.evidence_store
import ehr_pipeline.runtime
import ehr_pipeline.stages.s1_evidence
import ehr_pipeline.stages.s2_extract
import ehr_pipeline.stages.s3_verify
import ehr_pipeline.stages.s4_context
import ehr_pipeline.stages.s5_fact_sheet
import ehr_pipeline.stages.s6_summarize
import ehr_pipeline.stages.s7_check
import ehr_pipeline.stages.s8_review
import ehr_pipeline.extraction
import ehr_pipeline.verification
import ehr_pipeline.pipeline
import benchmarks.mimic
import benchmarks.metrics

for _mod in [
    ehr_pipeline.config, ehr_pipeline.prompts, ehr_pipeline.ollama_client,
    ehr_pipeline.schemas, ehr_pipeline.evidence_store, ehr_pipeline.runtime,
    ehr_pipeline.stages.s1_evidence, ehr_pipeline.stages.s2_extract,
    ehr_pipeline.stages.s3_verify, ehr_pipeline.stages.s4_context,
    ehr_pipeline.stages.s5_fact_sheet, ehr_pipeline.stages.s6_summarize,
    ehr_pipeline.stages.s7_check, ehr_pipeline.stages.s8_review,
    ehr_pipeline.extraction, ehr_pipeline.verification, ehr_pipeline.pipeline,
    benchmarks.mimic, benchmarks.metrics,
]:
    importlib.reload(_mod)

from ehr_pipeline import config as pipeline_config
from ehr_pipeline.pipeline import run_pipeline
from benchmarks import mimic, metrics

_CITATION_RE = re.compile(r"\[E:[^\]]+\]")


def strip_citations_for_metrics(text: str) -> str:
    """Remove evidence citations before benchmark scoring and compression counts."""
    return _CITATION_RE.sub("", text).strip()


print("project root:", PROJECT_ROOT)
print("models:")
for k, v in vars(pipeline_config.MODELS).items():
    print(f"  {k}: {v}")
print("ollama host:", pipeline_config.OLLAMA_HOST)
print("api key set:", bool(pipeline_config.OLLAMA_API_KEY))

project root: /Users/natejly/Documents/GitHub/LLMS
models:
  claim_extraction: gemma4:31b
  claim_verification: gemma4:31b
  context_agent: gemma4:31b
  summary_generation: gemma4:31b
  final_review: gemma4:31b
ollama host: https://ollama.com
api key set: True


## Configuration

In [2]:
EHR_JSON_PATH = PROJECT_ROOT / "datasets" / "bquxjob_5dbe3bd2_19dae30f3cf.json"
NOTES_CSV_PATH = PROJECT_ROOT / "datasets" / "mimic-iii-ext-notes-1.0.0" / "notes.csv"
LABELS_CSV_PATH = PROJECT_ROOT / "datasets" / "mimic-iii-ext-notes-1.0.0" / "labels.csv"

BENCH_DIR = PROJECT_ROOT / "outputs" / "_bench"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

NUM_CASES = 100
RANDOM_SEED = 7
ALLOW_VIOLATIONS = True       # don't drop a case just because the deterministic check found something
RESUME = True                 # reuse cached stage outputs across re-runs
ENABLE_CONTEXT_AGENT = False  # skip stage 4 in benchmark mode
SUMMARY_MIN_RATIO = 0.20
SUMMARY_MAX_RATIO = 0.30
BERTSCORE_MODEL = "roberta-large"

# Ollama: if OLLAMA_HOST is http://localhost:11434, do NOT send OLLAMA_API_KEY
# in the Authorization header (local Ollama returns 401). Keys are only used
# when OLLAMA_HOST=https://ollama.com unless you set OLLAMA_SEND_BEARER_TO_LOCAL=1.

print("work dir:", BENCH_DIR)
print("context agent:", "ON" if ENABLE_CONTEXT_AGENT else "OFF")
print(f"compression target: {SUMMARY_MIN_RATIO:.0%}-{SUMMARY_MAX_RATIO:.0%}")

work dir: /Users/natejly/Documents/GitHub/LLMS/outputs/_bench
context agent: OFF
compression target: 20%-30%


## Load and sample real cases (notes joined with structured EHR)

In [3]:
import random

all_cases = mimic.load_real_cases(
    ehr_json_path=EHR_JSON_PATH,
    notes_csv_path=NOTES_CSV_PATH,
    labels_csv_path=LABELS_CSV_PATH,
)
print(f"loaded {len(all_cases)} cases (notes joined with EHR by hadm_id)")

rng = random.Random(RANDOM_SEED)
sampled = rng.sample(all_cases, k=min(NUM_CASES, len(all_cases)))
for c in sampled:
    print(
        f"  {c.case_id:36s}  note_chars={len(c.note_text):5d}  "
        f"gold_concepts={len(c.gold_concepts):3d}  "
        f"diags={len(c.admission.get('all_diagnoses') or []):3d}  "
        f"procs={len(c.admission.get('all_procedures') or []):3d}"
    )

loaded 150 cases (notes joined with EHR by hadm_id)
  mimic-72907-165405-520384             note_chars= 3404  gold_concepts=  9  diags= 18  procs= 16
  mimic-29307-175627-318233             note_chars=  784  gold_concepts=  7  diags= 14  procs=  4
  mimic-47927-161682-717576             note_chars= 2592  gold_concepts=  7  diags= 18  procs=  5
  mimic-77013-141363-444743             note_chars= 2827  gold_concepts=  7  diags= 44  procs= 28
  mimic-54229-165594-598031             note_chars= 2337  gold_concepts=  9  diags= 48  procs= 16
  mimic-17539-153621-390912             note_chars= 4686  gold_concepts=  5  diags= 22  procs= 10
  mimic-85464-164726-466035             note_chars= 3032  gold_concepts=  9  diags= 34  procs=  6
  mimic-79877-185698-417636             note_chars= 1836  gold_concepts=  8  diags= 14  procs= 10
  mimic-94229-146659-541449             note_chars= 2700  gold_concepts=  6  diags= 35  procs=  9
  mimic-27022-196712-357513             note_chars= 1728  gold_con

## Materialize FHIR + notes and run the pipeline

In [ ]:
import traceback

from ehr_pipeline.ollama_client import OllamaError

# Reset the Ollama client singleton so any config/env changes take effect.
from ehr_pipeline import ollama_client as _oc
_oc._client = None
_oc._client_fingerprint = None

# Stage labels for the debug table (format placeholders filled at runtime).
_STAGE_LABELS = {
    "stage1_evidence":   "S1  Evidence store      [deterministic]",
    "stage2_extract":    "S2  Claim extraction    [LLM: {claim_extraction}]",
    "stage3_verify":     "S3  Claim verification  [LLM: {claim_verification}]",
    "stage4_context":    "S4  Context agent       [LLM: {context_agent}]",
    "stage5_fact_sheet": "S5  Fact sheet          [deterministic]",
    "stage6_summarize":  "S6  Summary generation  [LLM: {summary_generation}]",
    "stage7_check":      "S7  Deterministic check [deterministic]",
    "stage8_review":     "S8  Final LLM review    [LLM: {final_review}]",
}

def _print_stage_debug(pipe_result):
    """Print a per-stage timing table for one pipeline run."""
    m = {
        "claim_extraction":   pipeline_config.MODELS.claim_extraction,
        "claim_verification": pipeline_config.MODELS.claim_verification,
        "context_agent":      pipeline_config.MODELS.context_agent,
        "summary_generation": pipeline_config.MODELS.summary_generation,
        "final_review":       pipeline_config.MODELS.final_review,
    }
    W = 60
    print(f"  ┌{'─'*W}┬──────────┬────────┐")
    print(f"  │ {'Stage':<{W-2}} │  Time(s) │ Status │")
    print(f"  ├{'─'*W}┼──────────┼────────┤")
    for t in pipe_result.timings:
        label = _STAGE_LABELS.get(t.name, t.name).format(**m)
        status = "SKIP " if t.skipped else "ok   "
        print(f"  │ {label:<{W-2}} │ {t.seconds:>8.2f} │ {status:<6} │")
    total = sum(t.seconds for t in pipe_result.timings)
    print(f"  ├{'─'*W}┼──────────┼────────┤")
    print(f"  │ {'TOTAL':<{W-2}} │ {total:>8.2f} │        │")
    print(f"  └{'─'*W}┴──────────┴────────┘")
    comp = pipe_result.compression
    if comp:
        band = "IN BAND ✓" if comp.in_target_band else "OUT OF BAND ✗"
        print(f"  compression: {comp.achieved_ratio:.2f}  "
              f"({comp.summary_chars} / {comp.note_chars} chars)  [{band}]")
    # Deterministic check result
    s7 = "PASS ✓" if pipe_result.check_passed else "FAIL ✗"
    print(f"  S7 det. check : {s7}")

    # LLM review result
    cc = pipe_result.review_concern_counts
    s8 = "PASS ✓" if pipe_result.review_passed else "FAIL ✗ (high-severity concern)"
    print(f"  S8 LLM review : {s8}  "
          f"(high={cc.get('high',0)}  medium={cc.get('medium',0)}  low={cc.get('low',0)})")
    for c in (getattr(pipe_result, '_review', None) and [] or []):
        pass  # concerns printed below if desired

    if pipe_result.input_note_path:
        print(f"  input note    : {pipe_result.input_note_path}")
    if pipe_result.summary_path:
        print(f"  summary       : {pipe_result.summary_path}")
    if pipe_result.revised_summary_path:
        rcp = pipe_result.revised_check_passed
        rcp_str = "check PASS ✓" if rcp else ("check FAIL ✗" if rcp is False else "not checked")
        print(f"  revised summ. : {pipe_result.revised_summary_path}  [{rcp_str}]")
    else:
        print(f"  revised summ. : (none — no applicable revisions)")


def _strip_citations_for_metrics(text: str) -> str:
    return strip_citations_for_metrics(text)


def _benchmark_compression(summary_text: str, note_chars: int) -> tuple[int, float, bool]:
    safe_note_chars = max(int(note_chars), 1)
    summary_chars = len(summary_text)
    ratio = summary_chars / safe_note_chars
    return summary_chars, ratio, SUMMARY_MIN_RATIO <= ratio <= SUMMARY_MAX_RATIO


def _load_cached_result(case, out_dir: Path):
    """Try to reconstruct a benchmark result dict from cached pipeline outputs.

    Returns the result dict if audit.json exists (i.e. the pipeline ran to
    completion for this case), otherwise returns None.
    """
    audit_path = out_dir / "audit.json"
    if not audit_path.exists():
        return None

    import json as _j
    audit = _j.loads(audit_path.read_text("utf-8"))

    # Determine which summary to use as the prediction.
    summary_path = out_dir / "summary.md"
    revised_path = out_dir / "summary_revised.md"

    comp = audit.get("compression", {})
    check = audit.get("check_report", {})
    review = audit.get("review", {})
    cc = audit.get("review_concern_counts", {})
    ps = audit.get("patient_summary")
    revised_check = audit.get("revised_check_passed")

    if audit.get("stage_failed"):
        prediction = ""
        prediction_source = "none"
    elif revised_path.exists() and revised_check:
        prediction = revised_path.read_text("utf-8")
        prediction_source = "revised"
    elif summary_path.exists():
        prediction = summary_path.read_text("utf-8")
        prediction_source = "original"
    else:
        prediction = ""
        prediction_source = "none"

    patient_md_path = out_dir / "patient_summary.md"
    patient_md = patient_md_path.read_text("utf-8") if patient_md_path.exists() else ""

    timings = audit.get("timings", [])
    total_seconds = sum(t.get("seconds", 0) for t in timings)

    reference = mimic.real_case_reference_summary(case)
    gold_set = mimic.real_case_gold_entities(case)
    note_chars = comp.get("note_chars", len(case.note_text))
    prediction = _strip_citations_for_metrics(prediction)
    summary_chars, compression_ratio, compression_in_band = _benchmark_compression(prediction, note_chars)

    return {
        "case_id": case.case_id,
        "check_passed": check.get("passed", False) if check else (audit.get("stage_failed") is None),
        "review_passed": review.get("passed", True) if review else True,
        "review_high": cc.get("high", 0),
        "review_medium": cc.get("medium", 0),
        "review_low": cc.get("low", 0),
        "prediction_source": prediction_source,
        "context_agent_on": audit.get("context_agent_enabled", ENABLE_CONTEXT_AGENT),
        "prediction": prediction,
        "reference": reference,
        "gold_entities": gold_set,
        "output_dir": str(out_dir),
        "total_seconds": total_seconds,
        "note_chars": note_chars,
        "summary_chars": summary_chars,
        "compression_ratio": compression_ratio,
        "compression_in_band": compression_in_band,
        "patient_summary": patient_md,
        "patient_fk_grade": ps.get("fk_grade") if ps else None,
        "patient_fk_words": ps.get("fk_words", 0) if ps else 0,
        "patient_fk_sentences": ps.get("fk_sentences", 0) if ps else 0,
        "patient_chars": ps.get("chars", 0) if ps else 0,
        "patient_in_target_band": ps.get("in_target_band", False) if ps else False,
    }


results = []
cached_count = 0
for case in sampled:
    bundle_path, notes_dir = mimic.materialize_real_case(case, BENCH_DIR)
    out_dir = pipeline_config.output_dir(case.case_id)

    print(f"\n{'='*66}")
    print(f"  CASE: {case.case_id}")
    print(f"  note_chars={len(case.note_text)}  gold_concepts={len(case.gold_concepts)}")
    print(f"{'='*66}")

    # Skip cases that already have a completed pipeline run on disk.
    cached = _load_cached_result(case, out_dir)
    if cached is not None:
        cached_count += 1
        band = "IN BAND ✓" if cached["compression_in_band"] else "OUT OF BAND ✗"
        print(f"  *** CACHED — skipping pipeline (audit.json exists) ***")
        print(f"  compression: {cached['compression_ratio']:.2f}  "
              f"({cached['summary_chars']} / {cached['note_chars']} chars)  [{band}]")
        print(f"  prediction source: {cached['prediction_source']}")
        results.append(cached)
        continue

    try:
        pipe_result = run_pipeline(
            case_id=case.case_id,
            bundle_path=bundle_path,
            notes_dir=notes_dir,
            resume=RESUME,
            allow_violations=ALLOW_VIOLATIONS,
            enable_context_agent=ENABLE_CONTEXT_AGENT,
            summary_min_ratio=SUMMARY_MIN_RATIO,
            summary_max_ratio=SUMMARY_MAX_RATIO,
        )
    except OllamaError as exc:
        print(f"  ERROR: {exc}")
        if exc.status_code == 401:
            print("  Hint: local Ollama rejects Bearer tokens — set OLLAMA_HOST=https://ollama.com"
                  " or unset OLLAMA_API_KEY for local runs.")
        elif exc.status_code == 404:
            print("  Hint: model not found. Check MODELS in ehr_pipeline/config.py.")
        traceback.print_exc()
        reference = mimic.real_case_reference_summary(case)
        gold_set = mimic.real_case_gold_entities(case)
        results.append({
            "case_id": case.case_id,
            "check_passed": False,
            "context_agent_on": ENABLE_CONTEXT_AGENT,
            "prediction": "",
            "reference": reference,
            "gold_entities": gold_set,
            "output_dir": str(pipeline_config.output_dir(case.case_id)),
            "total_seconds": 0.0,
            "note_chars": len(case.note_text),
            "summary_chars": 0,
            "compression_ratio": 0.0,
            "compression_in_band": False,
            "error": str(exc),
        })
        continue

    _print_stage_debug(pipe_result)

    if pipe_result.summary_path is None:
        print(f"\n  WARNING: no summary emitted (check_passed=False, "
              f"audit at {pipe_result.audit_path})")
        prediction = ""
        prediction_source = "none"
    elif pipe_result.revised_summary_path and pipe_result.revised_check_passed:
        # Use the revised summary when it exists AND passes the deterministic check.
        prediction = pipe_result.revised_summary_path.read_text(encoding="utf-8")
        prediction_source = "revised"
    else:
        prediction = pipe_result.summary_path.read_text(encoding="utf-8")
        prediction_source = "original"

    prediction = _strip_citations_for_metrics(prediction)
    print(f"  benchmarking  : using {prediction_source} summary for metrics (citations stripped)")

    reference = mimic.real_case_reference_summary(case)
    gold_set = mimic.real_case_gold_entities(case)

    comp = pipe_result.compression
    note_chars = comp.note_chars if comp else len(case.note_text)
    summary_chars, compression_ratio, compression_in_band = _benchmark_compression(prediction, note_chars)
    cc = pipe_result.review_concern_counts
    ps = pipe_result.patient_summary
    patient_md = ps.path.read_text(encoding="utf-8") if ps and ps.path.exists() else ""
    results.append({
        "case_id": case.case_id,
        "check_passed": pipe_result.check_passed,
        "review_passed": pipe_result.review_passed,
        "review_high": cc.get("high", 0),
        "review_medium": cc.get("medium", 0),
        "review_low": cc.get("low", 0),
        "prediction_source": prediction_source,   # "original" | "revised" | "none"
        "context_agent_on": pipe_result.context_agent_enabled,
        "prediction": prediction,
        "reference": reference,
        "gold_entities": gold_set,
        "output_dir": str(pipe_result.output_dir),
        "total_seconds": sum(t.seconds for t in pipe_result.timings),
        "note_chars": note_chars,
        "summary_chars": summary_chars,
        "compression_ratio": compression_ratio,
        "compression_in_band": compression_in_band,
        # Stage 9 — patient-facing summary (independent text + readability score)
        "patient_summary": patient_md,
        "patient_fk_grade": ps.fk_grade if ps else None,
        "patient_fk_words": ps.fk_words if ps else 0,
        "patient_fk_sentences": ps.fk_sentences if ps else 0,
        "patient_chars": ps.chars if ps else 0,
        "patient_in_target_band": ps.in_target_band if ps else False,
    })

print(f"\n{'='*66}")
print(f"  DONE: {len(results)} cases  ({cached_count} cached, {len(results) - cached_count} fresh)")
print(f"{'='*66}")

2026-05-03 03:04:49,318 INFO ehr_pipeline.stages.s1_evidence: Stage 1: building evidence store for case mimic-72907-165405-520384
2026-05-03 03:04:49,322 INFO ehr_pipeline.stages.s1_evidence:   wrote 46 evidence items (35 structured, 11 note sentences) to /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-72907-165405-520384/evidence_store.json
2026-05-03 03:04:49,322 INFO ehr_pipeline.runtime:   stage1_evidence done in 0.00s
2026-05-03 03:04:49,323 INFO ehr_pipeline.stages.s2_extract: Stage 2: extracting claims from notes



  CASE: mimic-72907-165405-520384
  note_chars=3404  gold_concepts=9


2026-05-03 03:07:59,037 INFO ehr_pipeline.stages.s2_extract:   extracted 59 claims, wrote /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-72907-165405-520384/claims.json
2026-05-03 03:07:59,040 INFO ehr_pipeline.runtime:   stage2_extract done in 189.72s
2026-05-03 03:07:59,041 INFO ehr_pipeline.stages.s3_verify: Stage 3: verifying 59 claims
2026-05-03 03:08:36,760 INFO ehr_pipeline.stages.s3_verify:   {'verified': 17, 'contradicted': 0, 'unsupported': 42} -> /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-72907-165405-520384/verifications.json
2026-05-03 03:08:36,766 INFO ehr_pipeline.runtime:   stage3_verify done in 37.73s
2026-05-03 03:08:36,768 INFO ehr_pipeline.verification: Stage 4 (context agent) disabled by caller; using raw verifications
2026-05-03 03:08:36,797 INFO ehr_pipeline.stages.s5_fact_sheet: Stage 5: compiling verified fact sheet
2026-05-03 03:08:36,850 INFO ehr_pipeline.stages.s5_fact_sheet:   fact sheet sections: {'hpi': 0, 'active_problems': 48, 'medications':

  ┌────────────────────────────────────────────────────────────┬──────────┬────────┐
  │ Stage                                                      │  Time(s) │ Status │
  ├────────────────────────────────────────────────────────────┼──────────┼────────┤
  │ S1  Evidence store      [deterministic]                    │     0.00 │ ok     │
  │ S2  Claim extraction    [LLM: gemma4:31b]                  │   189.72 │ ok     │
  │ S3  Claim verification  [LLM: gemma4:31b]                  │    37.73 │ ok     │
  │ S4  Context agent       [LLM: gemma4:31b]                  │     0.00 │ SKIP   │
  │ S5  Fact sheet          [deterministic]                    │     0.05 │ ok     │
  │ stage9_patient_summary                                     │    10.26 │ ok     │
  │ S6  Summary generation  [LLM: gemma4:31b]                  │    14.04 │ ok     │
  │ S7  Deterministic check [deterministic]                    │     0.75 │ ok     │
  │ S8  Final LLM review    [LLM: gemma4:31b]                  │ 

2026-05-03 03:11:02,265 INFO ehr_pipeline.stages.s2_extract:   extracted 23 claims, wrote /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-29307-175627-318233/claims.json
2026-05-03 03:11:02,270 INFO ehr_pipeline.runtime:   stage2_extract done in 106.40s
2026-05-03 03:11:02,271 INFO ehr_pipeline.stages.s3_verify: Stage 3: verifying 23 claims
2026-05-03 03:12:19,349 INFO ehr_pipeline.stages.s3_verify:   {'verified': 19, 'contradicted': 0, 'unsupported': 4} -> /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-29307-175627-318233/verifications.json
2026-05-03 03:12:19,353 INFO ehr_pipeline.runtime:   stage3_verify done in 77.08s
2026-05-03 03:12:19,353 INFO ehr_pipeline.verification: Stage 4 (context agent) disabled by caller; using raw verifications
2026-05-03 03:12:19,356 INFO ehr_pipeline.stages.s5_fact_sheet: Stage 5: compiling verified fact sheet
2026-05-03 03:12:19,357 INFO ehr_pipeline.stages.s5_fact_sheet:   fact sheet sections: {'hpi': 0, 'active_problems': 28, 'medications': 

...

  ┌────────────────────────────────────────────────────────────┬──────────┬────────┐
  │ Stage                                                      │  Time(s) │ Status │
  ├────────────────────────────────────────────────────────────┼──────────┼────────┤
  │ S1  Evidence store      [deterministic]                    │     0.00 │ ok     │
  │ S2  Claim extraction    [LLM: gemma4:31b]                  │   123.73 │ ok     │
  │ S3  Claim verification  [LLM: gemma4:31b]                  │   146.08 │ ok     │
  │ S4  Context agent       [LLM: gemma4:31b]                  │     0.00 │ SKIP   │
  │ S5  Fact sheet          [deterministic]                    │     0.00 │ ok     │
  │ stage9_patient_summary                                     │    17.82 │ ok     │
  │ S6  Summary generation  [LLM: gemma4:31b]                  │    41.73 │ ok     │
  │ S7  Deterministic check [deterministic]                    │     0.00 │ ok     │
  │ S8  Final LLM review    [LLM: gemma4:31b]                  │ 

2026-05-03 16:03:09,736 INFO ehr_pipeline.stages.s2_extract:   extracted 22 claims, wrote /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-68135-163192-486433/claims.json
2026-05-03 16:03:09,737 INFO ehr_pipeline.runtime:   stage2_extract done in 81.72s
2026-05-03 16:03:09,738 INFO ehr_pipeline.stages.s3_verify: Stage 3: verifying 22 claims
2026-05-03 16:05:50,547 INFO ehr_pipeline.stages.s3_verify:   {'verified': 12, 'contradicted': 0, 'unsupported': 10} -> /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-68135-163192-486433/verifications.json
2026-05-03 16:05:50,548 INFO ehr_pipeline.runtime:   stage3_verify done in 160.81s
2026-05-03 16:05:50,548 INFO ehr_pipeline.verification: Stage 4 (context agent) disabled by caller; using raw verifications
2026-05-03 16:05:50,550 INFO ehr_pipeline.stages.s5_fact_sheet: Stage 5: compiling verified fact sheet
2026-05-03 16:05:50,550 INFO ehr_pipeline.stages.s5_fact_sheet:   fact sheet sections: {'hpi': 0, 'active_problems': 37, 'medications':

  ┌────────────────────────────────────────────────────────────┬──────────┬────────┐
  │ Stage                                                      │  Time(s) │ Status │
  ├────────────────────────────────────────────────────────────┼──────────┼────────┤
  │ S1  Evidence store      [deterministic]                    │     0.00 │ ok     │
  │ S2  Claim extraction    [LLM: gemma4:31b]                  │    81.72 │ ok     │
  │ S3  Claim verification  [LLM: gemma4:31b]                  │   160.81 │ ok     │
  │ S4  Context agent       [LLM: gemma4:31b]                  │     0.00 │ SKIP   │
  │ S5  Fact sheet          [deterministic]                    │     0.00 │ ok     │
  │ stage9_patient_summary                                     │    14.28 │ ok     │
  │ S6  Summary generation  [LLM: gemma4:31b]                  │     8.01 │ ok     │
  │ S7  Deterministic check [deterministic]                    │     0.00 │ ok     │
  │ S8  Final LLM review    [LLM: gemma4:31b]                  │ 

2026-05-03 16:09:57,870 INFO ehr_pipeline.stages.s2_extract:   extracted 53 claims, wrote /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-9461-156692-597/claims.json
2026-05-03 16:09:57,872 INFO ehr_pipeline.runtime:   stage2_extract done in 136.42s
2026-05-03 16:09:57,872 INFO ehr_pipeline.stages.s3_verify: Stage 3: verifying 53 claims
2026-05-03 16:13:13,489 INFO ehr_pipeline.stages.s3_verify:   {'verified': 34, 'contradicted': 0, 'unsupported': 19} -> /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-9461-156692-597/verifications.json
2026-05-03 16:13:13,490 INFO ehr_pipeline.runtime:   stage3_verify done in 195.62s
2026-05-03 16:13:13,491 INFO ehr_pipeline.verification: Stage 4 (context agent) disabled by caller; using raw verifications
2026-05-03 16:13:13,492 INFO ehr_pipeline.stages.s5_fact_sheet: Stage 5: compiling verified fact sheet
2026-05-03 16:13:13,493 INFO ehr_pipeline.stages.s5_fact_sheet:   fact sheet sections: {'hpi': 0, 'active_problems': 32, 'medications': 9, 'la

  ┌────────────────────────────────────────────────────────────┬──────────┬────────┐
  │ Stage                                                      │  Time(s) │ Status │
  ├────────────────────────────────────────────────────────────┼──────────┼────────┤
  │ S1  Evidence store      [deterministic]                    │     0.01 │ ok     │
  │ S2  Claim extraction    [LLM: gemma4:31b]                  │   136.42 │ ok     │
  │ S3  Claim verification  [LLM: gemma4:31b]                  │   195.62 │ ok     │
  │ S4  Context agent       [LLM: gemma4:31b]                  │     0.00 │ SKIP   │
  │ S5  Fact sheet          [deterministic]                    │     0.00 │ ok     │
  │ stage9_patient_summary                                     │    39.87 │ ok     │
  │ S6  Summary generation  [LLM: gemma4:31b]                  │    17.49 │ ok     │
  │ S7  Deterministic check [deterministic]                    │     0.10 │ ok     │
  │ S8  Final LLM review    [LLM: gemma4:31b]                  │ 

## Recover results from cache (run this if the kernel crashed)

If the benchmark loop was interrupted, run the cell below to reload all
completed cases from their cached `audit.json` outputs. It rebuilds the
`results` list so the scoring cells can proceed normally.

In [19]:
results = []
recovered = 0
skipped = 0
for case in sampled:
    out_dir = pipeline_config.output_dir(case.case_id)
    cached = _load_cached_result(case, out_dir)
    if cached is not None:
        results.append(cached)
        recovered += 1
    else:
        skipped += 1

print(f"Recovered {recovered} cached results, {skipped} cases have no cached output.")
if results:
    print(f"First case: {results[0]['case_id']}  compression={results[0]['compression_ratio']:.2f}")
print("Run the scoring cells below to compute metrics.")

Recovered 100 cached results, 0 cases have no cached output.
First case: mimic-72907-165405-520384  compression=0.42
Run the scoring cells below to compute metrics.


## Inspect one generated summary vs reference

In [20]:
if results:
    r = results[0]
    print(f"--- {r['case_id']} ---\n")
    print(
        f"compression: {r['summary_chars']}/{r['note_chars']} = {r['compression_ratio']:.2f}"
        f" (target {SUMMARY_MIN_RATIO:.2f}-{SUMMARY_MAX_RATIO:.2f}, in_band={r['compression_in_band']})\n"
    )
    print("=== REFERENCE ===\n")
    print(r["reference"])
    print("\n=== PREDICTION ===\n")
    print(r["prediction"][:3000] or "(empty)")

--- mimic-72907-165405-520384 ---

compression: 1445/3404 = 0.42 (target 0.20-0.30, in_band=False)

=== REFERENCE ===

## HPI
77.0-year-old female.
## Active Problems
- Abdominal Pain.
- Sepsis.
- Septicemia.
- Pain.
- Grimaces.
- Respiratory Failure.
- Kidney Failure.
- Atrial Fibrillation.
## EHR Diagnoses
- Other postoperative infection.
- Disseminated candidiasis.
- Severe sepsis.
- Septic shock.
- Other suppurative peritonitis.
- Acute and subacute necrosis of liver.
- Acute kidney failure with lesion of tubular necrosis.
- Necrotizing fasciitis.
- Acute vascular insufficiency of intestine.
- None.
- Acidosis.
- Acute salpingitis and oophoritis.
- Candidiasis of other urogenital sites.
- Acute inflammatory diseases of uterus, except cervix.
- Unspecified essential hypertension.
- Depressive disorder, not elsewhere classified.
- Unspecified acquired hypothyroidism.
- Other specified surgical operations and procedures causing abnormal patient reaction, or later complication, without

## Compute ROUGE and entity metrics per case

In [23]:
rows = []
for r in results:
    prediction = strip_citations_for_metrics(r["prediction"])
    if not prediction.strip():
        continue
    note_chars = max(int(r.get("note_chars", 0)), 1)
    summary_chars = len(prediction)
    compression_ratio = summary_chars / note_chars
    row = {
        "case_id": r["case_id"],
        "check_passed": r["check_passed"],
        "context_agent_on": r["context_agent_on"],
        "compression_ratio": compression_ratio,
        "compression_in_band": SUMMARY_MIN_RATIO <= compression_ratio <= SUMMARY_MAX_RATIO,
        "total_seconds": r["total_seconds"],
    }
    row.update(metrics.rouge_scores(prediction, r["reference"]))
    row.update(metrics.entity_recall_precision(
        prediction=prediction,
        reference=r["reference"],
        gold_entities=r["gold_entities"],
    ))
    # Stage 9 — patient summary readability (Flesch-Kincaid grade target 7-8).
    # Re-score from the saved markdown so the metric matches what the UI shows.
    patient_md = r.get("patient_summary", "")
    if patient_md.strip():
        fk = metrics.flesch_kincaid_grade(patient_md)
        row["patient_fk_grade"] = fk["fk_grade"]
        row["patient_fk_in_target"] = fk["fk_in_target_band"]
        row["patient_fk_words"] = fk["fk_words"]
        row["patient_fk_sentences"] = fk["fk_sentences"]
        row["patient_chars"] = r.get("patient_chars", len(patient_md))
    else:
        row["patient_fk_grade"] = None
        row["patient_fk_in_target"] = False
        row["patient_fk_words"] = 0
        row["patient_fk_sentences"] = 0
        row["patient_chars"] = 0
    rows.append(row)

rouge_df = pd.DataFrame(rows)
rouge_df

2026-05-03 16:26:59,931 INFO absl: Using default tokenizer.
2026-05-03 16:26:59,990 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,020 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,025 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,037 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,060 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,070 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,077 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,081 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,090 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,103 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,110 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,113 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,121 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,127 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,131 INFO absl: Using default tokenizer.
2026-05-03 16:27:00,144 INFO absl: Using

,case_id,check_passed,context_agent_on,compression_ratio,compression_in_band,total_seconds,rouge1_f,rouge2_f,rougeL_f,entity_precision,...,entity_f1,entity_tp,entity_fp,entity_fn,entity_gold,patient_fk_grade,patient_fk_in_target,patient_fk_words,patient_fk_sentences,patient_chars
0,mimic-72907-165405-520384,True,False,0.227380,True,266.204719,0.431095,0.234875,0.275618,1.0,...,0.561404,16,0,25,41,7.13,True,166,17,1385
1,mimic-29307-175627-318233,True,False,0.369898,False,270.373864,0.191781,0.055556,0.150685,1.0,...,0.160000,2,0,21,23,6.51,False,177,18,1490
2,mimic-47927-161682-717576,True,False,0.233796,True,407.149022,0.408377,0.201058,0.314136,1.0,...,0.651163,14,0,15,29,5.63,False,187,28,1714
3,mimic-77013-141363-444743,True,False,0.247259,True,294.464266,0.168950,0.055046,0.109589,1.0,...,0.340426,8,0,31,39,8.70,True,216,25,2129
4,mimic-54229-165594-598031,True,False,0.301241,False,231.680493,0.214433,0.062112,0.115464,1.0,...,0.641509,17,0,19,36,6.10,False,205,24,2133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,mimic-83607-156758-631634,True,False,0.232527,True,354.648615,0.284900,0.103152,0.148148,1.0,...,0.566667,17,0,26,43,7.69,True,159,12,1351
96,mimic-25326-191645-386337,True,False,0.263584,True,184.428249,0.066815,0.031320,0.062361,1.0,...,0.235294,6,0,39,45,5.60,False,148,14,1367
97,mimic-56635-192782-481041,True,False,0.255663,True,356.589775,0.353887,0.134771,0.160858,1.0,...,0.515152,17,0,32,49,6.57,False,179,16,1729
98,mimic-68135-163192-486433,True,False,0.252120,True,353.303235,0.198413,0.064000,0.111111,1.0,...,0.307692,6,0,27,33,6.92,False,178,17,1494


## BERTScore (batched, downloads `roberta-large` the first time)

In [24]:
non_empty = [r for r in results if strip_citations_for_metrics(r["prediction"]).strip()]
preds = [strip_citations_for_metrics(r["prediction"]) for r in non_empty]
refs = [r["reference"] for r in non_empty]
case_ids = [r["case_id"] for r in non_empty]

if preds:
    bert_rows = metrics.bertscore_batch(preds, refs, model_type=BERTSCORE_MODEL)
    bert_df = pd.DataFrame(bert_rows, index=case_ids)
    bert_df.index.name = "case_id"
    bert_df = bert_df.reset_index()
else:
    bert_df = pd.DataFrame(columns=["case_id", "bertscore_p", "bertscore_r", "bertscore_f1"])

bert_df

2026-05-03 16:27:07,036 INFO httpx: HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-05-03 16:27:07,076 INFO httpx: HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-03 16:27:07,123 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-05-03 16:27:07,156 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-03 16:27:07,194 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-05-03 16:27:07,229 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main?recursive=true&expa

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,case_id,bertscore_p,bertscore_r,bertscore_f1
0,mimic-72907-165405-520384,0.834571,0.812012,0.823137
1,mimic-29307-175627-318233,0.818140,0.809801,0.813949
2,mimic-47927-161682-717576,0.852225,0.826957,0.839401
3,mimic-77013-141363-444743,0.787992,0.788629,0.788310
4,mimic-54229-165594-598031,0.798852,0.793593,0.796214
...,...,...,...,...
95,mimic-83607-156758-631634,0.825362,0.798174,0.811540
96,mimic-25326-191645-386337,0.825136,0.777330,0.800520
97,mimic-56635-192782-481041,0.822166,0.810080,0.816078
98,mimic-68135-163192-486433,0.839166,0.792648,0.815244


## Combined per-case metrics

In [25]:
if not rouge_df.empty and not bert_df.empty:
    full = rouge_df.merge(bert_df, on="case_id", how="left")
else:
    full = rouge_df.copy()

metric_cols = [c for c in (
    "rouge1_f", "rouge2_f", "rougeL_f",
    "bertscore_p", "bertscore_r", "bertscore_f1",
    "entity_precision", "entity_recall", "entity_f1",
    "compression_ratio",
    "patient_fk_grade",
) if c in full.columns]

display_cols = ["case_id", "check_passed", "context_agent_on", "compression_in_band",
                *metric_cols, "patient_fk_in_target",
                "entity_tp", "entity_fn", "entity_gold", "total_seconds"]
full[[c for c in display_cols if c in full.columns]].round(3)

,case_id,check_passed,context_agent_on,compression_in_band,rouge1_f,rouge2_f,rougeL_f,bertscore_p,bertscore_r,bertscore_f1,entity_precision,entity_recall,entity_f1,compression_ratio,patient_fk_grade,patient_fk_in_target,entity_tp,entity_fn,entity_gold,total_seconds
0,mimic-72907-165405-520384,True,False,True,0.431,0.235,0.276,0.835,0.812,0.823,1.0,0.390,0.561,0.227,7.13,True,16,25,41,266.205
1,mimic-29307-175627-318233,True,False,False,0.192,0.056,0.151,0.818,0.810,0.814,1.0,0.087,0.160,0.370,6.51,False,2,21,23,270.374
2,mimic-47927-161682-717576,True,False,True,0.408,0.201,0.314,0.852,0.827,0.839,1.0,0.483,0.651,0.234,5.63,False,14,15,29,407.149
3,mimic-77013-141363-444743,True,False,True,0.169,0.055,0.110,0.788,0.789,0.788,1.0,0.205,0.340,0.247,8.70,True,8,31,39,294.464
4,mimic-54229-165594-598031,True,False,False,0.214,0.062,0.115,0.799,0.794,0.796,1.0,0.472,0.642,0.301,6.10,False,17,19,36,231.680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,mimic-83607-156758-631634,True,False,True,0.285,0.103,0.148,0.825,0.798,0.812,1.0,0.395,0.567,0.233,7.69,True,17,26,43,354.649
96,mimic-25326-191645-386337,True,False,True,0.067,0.031,0.062,0.825,0.777,0.801,1.0,0.133,0.235,0.264,5.60,False,6,39,45,184.428
97,mimic-56635-192782-481041,True,False,True,0.354,0.135,0.161,0.822,0.810,0.816,1.0,0.347,0.515,0.256,6.57,False,17,32,49,356.590
98,mimic-68135-163192-486433,True,False,True,0.198,0.064,0.111,0.839,0.793,0.815,1.0,0.182,0.308,0.252,6.92,False,6,27,33,353.303


## Aggregate 

In [26]:
if metric_cols:
    agg = full[metric_cols].agg(["mean", "median", "min", "max"]).round(3)
    display(agg)
else:
    print("No metrics computed.")

,rouge1_f,rouge2_f,rougeL_f,bertscore_p,bertscore_r,bertscore_f1,entity_precision,entity_recall,entity_f1,compression_ratio,patient_fk_grade
mean,0.290,0.127,0.187,0.816,0.810,0.813,1.0,0.377,0.523,0.264,7.383
median,0.288,0.114,0.182,0.816,0.805,0.812,1.0,0.360,0.530,0.257,7.355
min,0.018,0.008,0.016,0.761,0.769,0.782,1.0,0.051,0.098,0.131,4.440
max,0.704,0.526,0.456,0.864,0.905,0.884,1.0,0.852,0.920,0.400,11.580


## Save benchmark outputs

In [27]:
import json as _json

report_path = BENCH_DIR / "benchmark_report.json"
report = {
    "models": {k: v for k, v in vars(pipeline_config.MODELS).items()},
    "context_agent_enabled": ENABLE_CONTEXT_AGENT,
    "compression_target": [SUMMARY_MIN_RATIO, SUMMARY_MAX_RATIO],
    "per_case": full.to_dict(orient="records"),
    "aggregate": (full[metric_cols].agg(["mean", "median", "min", "max"]).round(4).to_dict() if metric_cols else {}),
    "num_cases": len(full),
}
report_path.write_text(_json.dumps(report, indent=2, default=str), encoding="utf-8")
print("wrote", report_path)

csv_path = BENCH_DIR / "benchmark_metrics.csv"
full.to_csv(csv_path, index=False)
print("wrote", csv_path)

wrote /Users/natejly/Documents/GitHub/LLMS/outputs/_bench/benchmark_report.json
wrote /Users/natejly/Documents/GitHub/LLMS/outputs/_bench/benchmark_metrics.csv
